# 🚦 Vietnamese Traffic Intelligence System
## 📊 Phase 2: Exploratory Data Analysis (EDA) & Final Data Validation
**Mục tiêu:** Kiểm toán, phân tích thống kê chuyên sâu toàn bộ tập dữ liệu đã chuẩn hóa (`Data/Processed`) trước khi bước vào giai đoạn Huấn luyện Mô hình (Model Development).

In [1]:
import os
import glob
import json
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

processed_dir = r'd:\HocTap\Bien_bao\Data\Processed'
print('Data Processed directory:', processed_dir)

Data Processed directory: d:\HocTap\Bien_bao\Data\Processed


--- 
## 🔍 Câu hỏi 1: Phân bố 7 Class Biển Báo Giao Thông như thế nào?
Phân tích số lượng nhãn của 7 lớp biển báo qua các tập Train / Val / Test (chia theo nhóm tuyến đường `street_id`).

In [2]:
# Đọc báo cáo tổng hợp EDA
with open(r'd:\HocTap\Bien_bao\Data\Reports\eda_summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)

print('=== TRAFFIC SIGNS SUMMARY ===')
for k, v in summary['traffic_signs'].items():
    print(f'  {k}: {v}')

=== TRAFFIC SIGNS SUMMARY ===
  total_objects: 11000
  train_objects: 6238
  val_objects: 1948
  test_objects: 2814
  small_objects_pct: 76.05
  medium_objects_pct: 21.29
  large_objects_pct: 2.66
  median_bbox_px: 17.0
  mean_bbox_px: 25.64
  imbalance_ratio: 5.48


### 📈 Biểu đồ Phân bố Lớp Biển Báo
![Traffic Signs Distribution](../Data/Reports/eda_charts/01_traffic_signs_class_dist.png)

--- 
## 🔍 Câu hỏi 2: Phân bố Trạng thái Đèn Giao Thông (Red vs Green)
Kiểm tra tính cân bằng giữa đèn Đỏ và đèn Xanh.

In [3]:
print('=== TRAFFIC LIGHTS SUMMARY ===')
for k, v in summary['traffic_lights'].items():
    print(f'  {k}: {v}')

=== TRAFFIC LIGHTS SUMMARY ===
  total_objects: 4883
  red_count: 2612
  green_count: 2271
  red_pct: 53.49
  green_pct: 46.51


### 📈 Biểu đồ Phân bố Đèn Tín Hiệu & Kích Thước
![Traffic Lights Distribution](../Data/Reports/eda_charts/02_traffic_lights_dist.png)

--- 
## 🔍 Câu hỏi 3 & 6: BBox nhỏ đến mức nào? Dataset có thực sự là bài toán Small-Object Detection?
* Phân loại theo chuẩn COCO:
  * **Small Object (< 32px):** Chiếm **76.05%**!
  * **Medium Object (32 - 96px):** Chiếm **21.29%**.
  * **Large Object (> 96px):** Chỉ chiếm **2.66%**.
* Kích thước trung vị (Median BBox size): **17.0 pixels**.

### 📈 Biểu đồ Tỷ Lệ Vật Thể Nhỏ (Small Object Dominance)
![Small Object Analysis](../Data/Reports/eda_charts/03_small_object_analysis.png)

--- 
## 🔍 Câu hỏi 4: Đánh giá Mức độ Mất Cân Bằng Lớp (Class Imbalance)
* Imbalance Ratio trong tập Biển báo: **5.48 : 1** (giữa lớp phổ biến nhất: *Nguy hiểm* - 3,049 và ít nhất: *Cấm rẽ* - 556).
* Đánh giá: Mức mất cân bằng trung bình (Moderate Imbalance), hoàn toàn được kiểm soát tốt bởi hàm Task-Aligned Focal Loss trong YOLOv11.

--- 
## 🔍 Câu hỏi 5: Ma trận Chữ Số Đếm Ngược & Kiểm tra Tính Toàn Vẹn
![Countdown Digits Matrix](../Data/Reports/eda_charts/04_countdown_digits_matrix.png)

--- 
## 🎯 KẾT LUẬN & ĐỀ XUẤT CHO PHASE 3 (BASELINE MODEL SELECTION)

1. **Đóng băng Dữ liệu (Freeze Data/Processed):**
   * Toàn bộ 10,252 ảnh & nhãn đã đạt chuẩn 100%, không sửa đổi trong quá trình thử nghiệm.
2. **Lựa chọn Mô hình Baseline:**
   * **Biển báo:** `YOLO11n` (imgsz=640) làm baseline đối chứng.
   * **Đèn giao thông:** `YOLO11n` (imgsz=640).
   * **Chữ số đếm ngược:** Mạng CNN nhỏ / MobileNetV3 (CrossEntropyLoss).
3. **Chiến lược Cải tiến sau Baseline (Hypothesis-Driven):**
   * Tăng độ phân giải `imgsz=832` để kiểm chứng khả năng bắt vật thể nhỏ ($<17\text{px}$).
   * Đánh giá `mAP50`, `mAP50-95` và độ trễ suy luận (FPS) trước khi nâng lên `YOLO11s`.